In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
model_name = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import pandas as pd

# read data
reviews_df = pd.read_csv("Hotel_Reviews.csv")
df = reviews_df[["reviews.text", "reviews.rating"]].copy()

In [ ]:
df.head()

,reviews.text,reviews.rating
0,Our experience at Rancho Valencia was absolute...,5.0
1,Amazing place. Everyone was extremely warm and...,5.0
2,We booked a 3 night stay at Rancho Valencia to...,5.0
3,Currently in bed writing this for the past hr ...,2.0
4,I live in Md and the Aloft is my Home away fro...,5.0


In [ ]:
def create_label(rating):
    if rating <= 2:
        return 0
    elif rating >= 4:
        return 1
    return -1

In [ ]:
df["label"] = df["reviews.rating"].apply(create_label)

In [ ]:
df = df[df["label"] != -1]
df = df[["reviews.text", "label"]]

In [ ]:
df.head()

,reviews.text,label
0,Our experience at Rancho Valencia was absolute...,1
1,Amazing place. Everyone was extremely warm and...,1
2,We booked a 3 night stay at Rancho Valencia to...,1
3,Currently in bed writing this for the past hr ...,0
4,I live in Md and the Aloft is my Home away fro...,1


In [ ]:
from sklearn.model_selection import train_test_split

train_data,temp_data = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

test_data,val_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    stratify=temp_data["label"]
)

In [ ]:
# convert the pandas DataFrames into Hugging Face Dataset objects

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_data)
val_dataset = Dataset.from_pandas(val_data)
test_dataset = Dataset.from_pandas(test_data)

In [ ]:
print(val_dataset)

Dataset({
    features: ['reviews.text', 'label', '__index_level_0__'],
    num_rows: 844
})


In [ ]:
train_dataset = train_dataset.remove_columns(["__index_level_0__"])
val_dataset = val_dataset.remove_columns(["__index_level_0__"])
test_dataset = test_dataset.remove_columns(["__index_level_0__"])

In [ ]:
print(train_dataset[0])

{'reviews.text': "Great location oceanfront on Boardwalk. Staff was very friendly and accommodating. Room just need some updating and wasn't exactly clean. It just needs a more through cleaning than the everyday housekeeping and I think it would be back on track. This was our 3rd year staying here and will probably come again next year!", 'label': 1}


In [ ]:
train_dataset = train_dataset.rename_column("reviews.text", "text")
val_dataset = val_dataset.rename_column("reviews.text", "text")
test_dataset = test_dataset.rename_column("reviews.text", "text")

In [ ]:
sample = train_dataset[0]["text"]

tokens = tokenizer(sample)

print(tokens)

{'input_ids': [101, 2307, 3295, 4153, 12792, 2006, 29496, 1012, 3095, 2001, 2200, 5379, 1998, 16222, 5358, 5302, 16616, 1012, 2282, 2074, 2342, 2070, 2039, 16616, 1998, 2347, 1005, 1056, 3599, 4550, 1012, 2009, 2074, 3791, 1037, 2062, 2083, 9344, 2084, 1996, 10126, 2160, 18321, 1998, 1045, 2228, 2009, 2052, 2022, 2067, 2006, 2650, 1012, 2023, 2001, 2256, 3822, 2095, 6595, 2182, 1998, 2097, 2763, 2272, 2153, 2279, 2095, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/6750 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

Map:   0%|          | 0/844 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train[0])

{'text': "Great location oceanfront on Boardwalk. Staff was very friendly and accommodating. Room just need some updating and wasn't exactly clean. It just needs a more through cleaning than the everyday housekeeping and I think it would be back on track. This was our 3rd year staying here and will probably come again next year!", 'label': 1, 'input_ids': [101, 2307, 3295, 4153, 12792, 2006, 29496, 1012, 3095, 2001, 2200, 5379, 1998, 16222, 5358, 5302, 16616, 1012, 2282, 2074, 2342, 2070, 2039, 16616, 1998, 2347, 1005, 1056, 3599, 4550, 1012, 2009, 2074, 3791, 1037, 2062, 2083, 9344, 2084, 1996, 10126, 2160, 18321, 1998, 1045, 2228, 2009, 2052, 2022, 2067, 2006, 2650, 1012, 2023, 2001, 2256, 3822, 2095, 6595, 2182, 1998, 2097, 2763, 2272, 2153, 2279, 2095, 999, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sentiment_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.128472,0.958531,0.967655,0.984911,0.976207
2,0.169330,0.141242,0.960900,0.974114,0.980796,0.977444
3,0.079682,0.161939,0.958531,0.971467,0.980796,0.976109


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1266, training_loss=0.11062885724349421, metrics={'train_runtime': 271.2387, 'train_samples_per_second': 74.657, 'train_steps_per_second': 4.667, 'total_flos': 670616205696000.0, 'train_loss': 0.11062885724349421, 'epoch': 3.0})

In [ ]:
test_results = trainer.evaluate(tokenized_test)

print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.079682,0.108250,3,0.960900,0.980636,0.973901,0.977257


{'eval_loss': 0.10824979096651077, 'eval_accuracy': 0.9609004739336493, 'eval_precision': 0.9806362378976486, 'eval_recall': 0.9739010989010989, 'eval_f1': 0.9772570640937285}


In [ ]:
model_path = "./stayora-sentiment-model"

trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./stayora-sentiment-model/tokenizer_config.json',
 './stayora-sentiment-model/tokenizer.json')

In [ ]:
import os

print(os.listdir(model_path))

['training_args.bin', 'model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
model_path = "./stayora-sentiment-model"

test_tokenizer = AutoTokenizer.from_pretrained(model_path)
test_model = AutoModelForSequenceClassification.from_pretrained(model_path)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
def predict_sentiment(review):
    inputs = test_tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = test_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probabilities, dim=1).item()
    confidence = probabilities[0][prediction].item()

    sentiment = "positive" if prediction == 1 else "negative"

    return {
        "sentiment": sentiment,
        "confidence": confidence
    }

In [ ]:
print(predict_sentiment(
    "The hotel was clean, comfortable and the staff were wonderful."
))

tensor([[-3.4059,  3.0432]])
1
{'sentiment': 'positive', 'confidence': 0.9984205961227417}
